# Relaxation Settings

ASSYST provides several relaxation modes that differ in which degrees of freedom are minimized. They all share the same interface (see :class:`assyst.relaxations.Relax`) but apply different ASE filters and constraints to control whether positions, cell shape, volume, or symmetry are allowed to vary.

This notebook walks through each option on a small Cu structure using ASE's EMT calculator so the examples run quickly.

| Class | Positions | Cell shape | Volume | Symmetry |
|---|---|---|---|---|
| `Relax` | yes | fixed | fixed | free |
| `CellRelax` | fixed | yes | fixed | free |
| `VolumeRelax` | fixed | fixed | yes | free |
| `SymmetryRelax` | yes | yes | yes | preserved |
| `FullRelax` | yes | yes | yes | free |

In [ ]:
from assyst.relaxations import Relax, CellRelax, VolumeRelax, SymmetryRelax, FullRelax, relax

from ase.build import bulk
from ase.calculators.emt import EMT
import numpy as np

## Starting structure

We start from an FCC Cu cell that is slightly distorted away from its equilibrium so that all relaxation modes have something to do.

In [ ]:
def make_structure():
    s = bulk('Cu', 'fcc', a=3.7, cubic=True)
    # Shear and stretch the cell a little
    strain = np.eye(3) + np.array([[0.02, 0.01, 0.0], [0.01, -0.01, 0.0], [0.0, 0.0, 0.03]])
    s.set_cell(s.cell.array @ strain, scale_atoms=True)
    # Displace one atom
    s.positions[0] += [0.05, 0.0, 0.0]
    return s

s0 = make_structure()
print(f'volume   = {s0.get_volume():.3f} A^3')
print(f'cell =\n{s0.cell.array}')

Helper to run a single relaxation through :func:`assyst.relaxations.relax` and print a summary.

In [ ]:
def show(settings, label):
    [out] = list(relax([make_structure()], settings, EMT()))
    print(f'--- {label} ---')
    print(f'energy   = {out.get_potential_energy():.4f} eV')
    print(f'volume   = {out.get_volume():.3f} A^3')
    print(f'cell  =\n{out.cell.array}')
    print(f'pos[0]   = {out.positions[0]}')
    print()
    return out

## `Relax` -- positions only

The base class only relaxes internal positions. Cell shape and volume are kept fixed. This is the right choice if you want to study atomic relaxations at a fixed lattice (e.g. defect cores at a given volume).

In [ ]:
show(Relax(force_tolerance=1e-3), 'Relax (positions)')

## `VolumeRelax` -- isotropic volume only

`VolumeRelax` keeps relative atomic positions and the cell shape fixed and only scales the cell isotropically. An optional `pressure` (in eV/A^3) can be applied. Use this to find the equilibrium volume at fixed shape, e.g. for an EOS scan.

In [ ]:
show(VolumeRelax(force_tolerance=1e-3), 'VolumeRelax')

## `CellRelax` -- cell shape at fixed volume

`CellRelax` allows the cell shape to relax while the volume and the relative atomic positions are constrained. This is useful to determine an equilibrium cell shape independent of volume.

In [ ]:
show(CellRelax(force_tolerance=1e-3), 'CellRelax')

## `SymmetryRelax` -- full relaxation, preserving space group

`SymmetryRelax` relaxes positions and cell but uses ASE's `FixSymmetry` constraint to keep the initial space group of the structure. This is the workhorse of ASSYST's structure generation: it lets you reach the energy minimum of a given symmetry-constrained crystal without accidentally hopping into a different polymorph.

In [ ]:
show(SymmetryRelax(force_tolerance=1e-3), 'SymmetryRelax')

## `FullRelax` -- everything free

`FullRelax` relaxes positions and the full cell with no constraints. This may break the original symmetry of the structure but yields the true local minimum for the given calculator.

In [ ]:
show(FullRelax(force_tolerance=1e-3), 'FullRelax')

## Applying pressure

`VolumeRelax`, `SymmetryRelax`, and `FullRelax` accept a `pressure` argument (units of eV/A^3, the ASE convention). A positive pressure compresses the cell, a negative pressure expands it. This is convenient to generate structures off the equilibrium volume for the training set.

In [ ]:
for p in (-0.05, 0.0, 0.05):
    show(SymmetryRelax(force_tolerance=1e-3, pressure=p), f'SymmetryRelax @ p={p} eV/A^3')

## Optimizer and convergence

All relaxation classes share three knobs:

* `algorithm` -- `'LBFGS'` (default), `'BFGS'`, or `'FIRE'`. LBFGS is usually fastest for well-behaved systems; FIRE is more robust for noisy or stiff potentials.
* `force_tolerance` -- maximum residual force (eV/A) at which the optimizer stops.
* `max_steps` -- hard cap on the number of optimizer steps.


In [ ]:
for algo in ('LBFGS', 'BFGS', 'FIRE'):
    show(FullRelax(algorithm=algo, force_tolerance=1e-3, max_steps=200), f'FullRelax / {algo}')

## Relaxing many structures

`relax` is a generator: pass it any iterable of `Atoms` and it yields the relaxed copies one by one. Each output structure has a `SinglePointCalculator` attached with the final energy, forces, and stress, ready to be written to a training file.

In [ ]:
structures = [bulk('Cu', 'fcc', a=a, cubic=True) for a in (3.5, 3.6, 3.7, 3.8)]
settings = SymmetryRelax(force_tolerance=1e-3)
for s in relax(structures, settings, EMT()):
    print(f'a-equiv = {s.get_volume()**(1/3):.3f} A   E = {s.get_potential_energy():.4f} eV')